In [8]:
import os
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
load_dotenv()
api_key = os.getenv("API_KEY")

In [26]:
RunnablePassthrough().invoke([1, 2, 3])

[1, 2, 3]

In [9]:
chat_template_tools = ChatPromptTemplate.from_template('''
What are the five most important tools a {job title} needs?
Answer only by listing the tools.
''')

chat_template_strategy = ChatPromptTemplate.from_template('''
Considering the tools provided, develop a strategy for effectively learning and mastering them:
{tools}
''')

In [28]:
chat_template_tools

ChatPromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job title'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job title} needs?\nAnswer only by listing the tools.\n'), additional_kwargs={})])

In [10]:
chat = ChatOllama(
    model="gpt-oss:120b",
    temperature=0.7,
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {api_key}"
        }
    }
)

In [11]:
string_parser = StrOutputParser()

In [12]:
chain_tools = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()})
chain_strategy = chat_template_strategy | chat | string_parser

In [39]:
# Testing Each LCEL individually

In [40]:
print(chain_tools.invoke({'job title':'data scientist'})['tools'])

- Python  
- R  
- SQL  
- Jupyter Notebook  
- Git


In [34]:
print(chain_strategy.invoke({'tools':'''
1. Python
2. R Programming
3. SQL
4. Tableau
5. Hadoop
'''}))

['## 📚 Master‑Skill Blueprint for Python', 'R', 'SQL', 'Tableau & Hadoop  ', 'Below is a **step‑by‑step', '12‑month learning roadmap** that moves you from “zero‑to‑proficient” to “able to own end‑to‑end data‑science / analytics projects”.  ', 'It weaves together **theory', 'hands‑on practice', 'portfolio building', 'community', 'and certification** so you can demonstrate mastery to employers or clients.', '---', '## 1️⃣  OVERALL FRAMEWORK  ', '| Phase | Duration | Goal | Core Activities |', '|-------|----------|------|-----------------|', '| **A – Foundations** | 0‑2\u202fmo | Get comfortable with programming logic', 'data fundamentals', 'and basic visualization. | Intro courses', 'simple notebooks', 'SQL basics', 'Tableau “drag‑and‑drop”. |', '| **B – Intermediate Integration** | 3‑6\u202fmo | Combine tools', 'start building real‑world mini‑projects', 'learn data‑engineering basics. | APIs', 'data wrangling in Python/R', 'multi‑table SQL queries', 'Tableau storytelling', 'Hadoop ecosy

In [35]:
chain_combined = chain_tools | chain_strategy

In [36]:
print(chain_combined.invoke({'job title':'data scientist'}))

['## Learning & Mastery Road‑Map  ', '**Tools:** Python', 'Jupyter\u202fNotebook', 'pandas', 'scikit‑learn', 'Git  ', 'The strategy is organized into **four phases** (Foundation → Integration → Specialisation → Mastery) and includes **weekly milestones**', '**high‑impact resources**', '**hands‑on projects**', 'and **habit‑building practices**. Feel free to stretch or compress the timeline to match your schedule', 'but aim to spend **≈10\u202fh/week** for steady progress.', '---', '### 1️⃣ Phase\u202f0 – Set‑up the Learning Environment (1\u202fweek)', '| Action | Why | Quick‑Start Steps |', '|-------|-----|-------------------|', '| Install **conda** (or Miniconda) | Manages isolated Python environments for each project | `conda create -n ds python=3.11` → `conda activate ds` |', '| Install **JupyterLab** | Modern UI for notebooks', 'integrates with Git extensions | `conda install -c conda-forge jupyterlab` |', '| Install **Git** & create a **GitHub** account | Version‑control from day\u

In [13]:
chain_long = (chat_template_tools | chat | string_parser | {'tools':RunnablePassthrough()} | 
              chat_template_strategy | chat | string_parser)

In [14]:
chain_long.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       